
<img src="img/viu_logo.png" width="200">

## 01MIAR - Estructuras de datos, +Pandas

![logo](img/python_logo.png)

*Ivan Fuertes / Franklin Alvarez*

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Como leer este notebook (version mejorada)</b>

Este notebook fue **enriquecido** para ayudarte a aprender escribiendo codigo. Vas a encontrar tres tipos de bloques de colores:

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>AZUL = TEORIA</b></div> -> explicacion en palabras simples antes de cada concepto.

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>AMARILLO = TU TURNO</b></div> -> el ejercicio que TU debes resolver. Hay una celda vacia debajo donde escribes el codigo.

<div style="background-color:#ebfbee;border-left:6px solid #2f9e44;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>VERDE = SOLUCION</b></div> -> codigo de referencia comentado por si te trabas. Destapa solo despues de intentar.

> **Promesa**: las celdas y la teoria del profesor original NO se han tocado. Solo se han ANADIDO bloques. Asi puedes seguir la version original cuando quieras.

#### De que trata este notebook 10
Este es el segundo notebook de Pandas y se centra en **lo que hace a Pandas tan potente para analisis de datos**:

| Bloque | Idea clave |
|---|---|
| Operaciones | Aritmetica entre DataFrames y Series, alineacion automatica por etiquetas. |
| `apply` | Aplicar una funcion a cada fila o columna. |
| Estadistica descriptiva | `describe`, `mean`, `std`, `idxmax`... la chuleta del analista. |
| **Agregaciones (`groupby`)** | El concepto **mas importante** de Pandas: dividir, calcular y combinar. |
| Correlaciones | Relacion lineal entre variables numericas, base del feature engineering. |

</div>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pd.__version__

# Operaciones en pandas

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Operaciones entre DataFrames: alineacion y fill_value</b>

Cuando haces `frameA + frameB`, Pandas:

1. **Alinea** filas y columnas por sus etiquetas (no por posicion).
2. Donde la etiqueta existe en los dos -> calcula.
3. Donde NO coincide -> resultado `NaN`.

```python
A = pd.DataFrame({'x':[1,2], 'y':[3,4]})
B = pd.DataFrame({'x':[10,20], 'z':[100,200]})
A + B
#       x      y      z
# 0    11    NaN    NaN
# 1    22    NaN    NaN
```

#### Metodos en lugar de operadores: la ventaja del `fill_value`

| Operador | Metodo equivalente | Ventaja del metodo |
|---|---|---|
| `+` | `.add(other, fill_value=...)` | Rellenar huecos |
| `-` | `.sub(other, fill_value=...)` | idem |
| `*` | `.mul(other, fill_value=...)` | idem |
| `/` | `.div(other, fill_value=...)` | idem |
| `**` | `.pow(other, fill_value=...)` | idem |

```python
A.add(B, fill_value=0)   # cuando falta una etiqueta en uno de los dos, usa 0
```

> **Por que es util?** Imagina dos meses de ventas en DataFrames con productos distintos. Si haces `enero + febrero`, los productos que solo aparecen en uno daran NaN. Con `fill_value=0` cuentan como cero ventas el otro mes.

</div>

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Operaciones en Pandas: idea general</b>

Pandas combina dos cosas:

1. La **velocidad de NumPy** (operaciones vectorizadas en C, sin bucles Python).
2. Las **etiquetas** (nombres de filas y columnas) que te permiten trabajar de forma legible.

#### Operaciones vectorizadas
En Pandas, cuando haces `df + 1`, **se suma 1 a cada celda** sin que tu escribas un bucle. Esto es:

- **Mas rapido** que un `for` en Python (cientos a miles de veces).
- **Mas legible**: una linea expresa la intencion.
- **Mas seguro**: NumPy/Pandas se encargan de dimensiones, NaN, dtypes.

```python
df['precio'] * 1.21         # IVA aplicado a toda la columna
df['edad'] + df['hijos']    # suma columna a columna
df > 0                      # mascara booleana del mismo tamano
```

#### Alineacion por etiquetas
Cuando operas dos DataFrames, Pandas **alinea por nombre de fila/columna**, no por posicion. Si una etiqueta no existe en uno de los dos, el resultado es `NaN` (a menos que uses `fill_value`).

```python
A + B   # alinea por indice y columna; donde no coincide -> NaN
A.add(B, fill_value=0)   # rellena con 0 lo que falte en uno de los dos
```

> Esta es la diferencia clave con NumPy: NumPy se basa en posiciones, Pandas en etiquetas.

</div>

In [ ]:
rand_matrix = np.random.randint(6,size=(4,6))
frame = pd.DataFrame(rand_matrix , columns=list('ABCDEF'))
display(frame)

In [ ]:
# Pandas utiliza numpy por debajo. Podemos acceder a un nparray equivalente a través de values
print(frame.values)
print(type(frame.values))

#### Búsqueda

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Busqueda en DataFrames: in, isin, mascaras booleanas</b>

Tres formas tipicas de **buscar** en un DataFrame:

| Forma | Para que sirve | Ejemplo |
|---|---|---|
| `'colname' in df` | Ver si existe una **columna**. | `'edad' in df` |
| `df.isin([v1, v2])` | Mascara booleana: True donde el valor **esta en la lista**. | `df.isin([2, 3, 4])` |
| `df['col'] == valor` | Mascara booleana sobre **una columna**. | `df['edad'] > 30` |
| `df.query("...")` | Filtro por expresion tipo SQL. | `df.query("edad > 30")` |

#### Mascaras booleanas: el patron rey de Pandas
Una **mascara** es un DataFrame (o Serie) del mismo tamano lleno de `True`/`False`. La usas para:

```python
# 1. Filtrar filas
df[df['edad'] > 30]

# 2. Contar cuantos True hay (True == 1)
(df['edad'] > 30).sum()

# 3. Reemplazar valores que cumplan una condicion
df.loc[df['edad'] < 18, 'categoria'] = 'menor'

# 4. Combinar mascaras con &, |, ~  (NO uses and/or/not con Pandas)
df[(df['edad'] > 30) & (df['ciudad'] == 'Madrid')]
```

> **Cuidado:** con Pandas se usa `&`, `|`, `~` y los **parentesis son obligatorios** en cada condicion. `and`/`or` lanzan error porque trabajan con un solo bool, no con un Series.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 1</b>

Crea un DataFrame de 5 filas y 4 columnas (`A, B, C, D`) con enteros aleatorios entre 0 y 9.

1. Comprueba si la columna `B` existe.
2. Genera una mascara booleana de los valores **mayores que 5**.
3. Cuenta cuantos valores cumplen la condicion **por columna** (suma sobre `axis=0`).
4. Filtra las **filas** donde la columna `A` sea mayor que 5.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd
import numpy as np

np.random.seed(0)
df = pd.DataFrame(np.random.randint(0, 10, size=(5, 4)), columns=list('ABCD'))
display(df)



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
# import numpy as np
#
# np.random.seed(0)
# df = pd.DataFrame(np.random.randint(0, 10, size=(5, 4)), columns=list('ABCD'))
# display(df)
#
# # 1) existencia de columna
# print('B' in df)
#
# # 2) mascara booleana
# mask = df > 5
# display(mask)
#
# # 3) contar True por columna (axis=0 es el eje de filas, suma a lo largo de filas)
# print(mask.sum(axis=0))
#
# # 4) filtrar filas con A > 5
# display(df[df['A'] > 5])


In [ ]:
# El operador in busca en columnas (DataFrame como dic, busca en claves)
'E' in frame

In [ ]:
# Puedes buscar valores en el dataframe que pertenezcan a una lista con el método isin
lista = [2, 3, 4]
search_mask = frame.isin(lista)

display(search_mask) # --> mask de respuesta (valores que son 3 o 2)
display(frame[search_mask])

In [ ]:
# Contar el número de ocurrencias
# Por columnas
print(search_mask.sum())
print()

# Por filas
print(search_mask.sum(axis=1))
print()

# En total
# print("Se han encontrado {} ocurrencias".format(search_mask.sum().sum()))
print("Se han encontrado {} ocurrencias".format(search_mask.values.sum()))

#### Ordenación

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Ordenacion: sort_index vs sort_values</b>

Pandas separa dos cosas que en Excel solemos confundir:

| Metodo | Que ordena | Ejemplo |
|---|---|---|
| `sort_index()` | Ordena por el **indice** (etiquetas de fila o columna). | `df.sort_index()` |
| `sort_values()` | Ordena por **el contenido** de una columna o fila. | `df.sort_values(by='edad')` |

#### Argumentos clave (ambos los aceptan)

| Argumento | Que hace | Valores tipicos |
|---|---|---|
| `axis` | 0 = filas (defecto), 1 = columnas. | `axis=0` o `axis=1` |
| `ascending` | Sentido. | `True` (defecto) o `False` |
| `inplace` | Modifica el DataFrame original o devuelve copia. | `True` o `False` (defecto) |
| `kind` | Algoritmo (`quicksort`, `mergesort`, `stable`). | normalmente lo dejas. |
| `na_position` | Donde poner los NaN. | `'last'` (defecto) o `'first'` |

#### Ejemplos pragmaticos
```python
# Top 5 productos por precio
df.sort_values('precio', ascending=False).head(5)

# Ordenar por dos criterios (primero por ciudad, luego por edad)
df.sort_values(['ciudad', 'edad'])

# Ordenar columnas alfabeticamente
df.sort_index(axis=1)
```

> **Truco:** `sort_values(by=['a','b'], ascending=[True, False])` permite criterios mixtos.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 2</b>

Tienes este DataFrame:

```python
df = pd.DataFrame({
    'producto': ['pan', 'leche', 'queso', 'huevos', 'agua'],
    'precio':   [1.20,   0.95,    4.30,    2.50,     0.50],
    'stock':    [50,     30,      10,      40,       100]
})
```

1. Ordena las filas por **precio descendente**.
2. Ordena por **stock ascendente** y, si empatan, por **precio ascendente**.
3. Ordena las **columnas** alfabeticamente.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd

df = pd.DataFrame({
    'producto': ['pan', 'leche', 'queso', 'huevos', 'agua'],
    'precio':   [1.20,   0.95,    4.30,    2.50,     0.50],
    'stock':    [50,     30,      10,      40,       100]
})



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
#
# df = pd.DataFrame({
#     'producto': ['pan', 'leche', 'queso', 'huevos', 'agua'],
#     'precio':   [1.20,   0.95,    4.30,    2.50,     0.50],
#     'stock':    [50,     30,      10,      40,       100]
# })
#
# # 1) precio descendente
# display(df.sort_values('precio', ascending=False))
#
# # 2) stock asc, en empates precio asc
# display(df.sort_values(['stock', 'precio'], ascending=[True, True]))
#
# # 3) columnas alfabeticas
# display(df.sort_index(axis=1))


In [ ]:
from random import shuffle

rand_matrix = np.random.randint(20,size=(5,4))

indices = list(range(5))
indx_col = list('ABCD')

shuffle(indices) # mezcla indices
shuffle(indx_col) # mezcla indices

frame = pd.DataFrame(rand_matrix , columns=indx_col, index=indices)
display(frame)

In [ ]:
# ordenar por índice
frame.sort_index(ascending=True, inplace=True)
display(frame)

In [ ]:
#ordenar por columna
frame.sort_index(axis=1, ascending=True, inplace=True)
display(frame)

In [ ]:
# ordenar filas por valor en columna
frame.sort_values(by='A', ascending=True, inplace=True)
display(frame)

In [ ]:
# ordenar columnas por valor en fila
frame.sort_values(by=1, axis=1, ascending=True, inplace=True)
display(frame)

In [ ]:
# Imprimir, uno a uno, los valores de la columna 'C' de mayor a menor
for x in frame.sort_values(by='C', ascending=False)['C']:
    print(x)

#### Ranking
- Construir un ranking de valores

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Ranking: rank() y sus metodos de empate</b>

`rank()` te da la **posicion (1, 2, 3...)** de cada valor dentro de una columna o fila. Util para podios, percentiles y normalizacion.

#### Como trata los empates? (parametro `method`)

| `method` | Que hace en empate | Ejemplo: valores `[10, 20, 20, 30]` -> ranks |
|---|---|---|
| `'average'` (defecto) | Promedio de las posiciones | `1, 2.5, 2.5, 4` |
| `'min'` | Posicion mas pequena | `1, 2, 2, 4` |
| `'max'` | Posicion mas grande | `1, 3, 3, 4` |
| `'first'` | Por orden de aparicion | `1, 2, 3, 4` |
| `'dense'` | Como `min`, pero sin huecos | `1, 2, 2, 3` |

#### Otros parametros
- `ascending=True/False`: 1 al menor o al mayor.
- `axis=0/1`: ranquear por columna (defecto) o por fila.
- `pct=True`: devuelve **percentil** en lugar de posicion absoluta.

```python
df['posicion'] = df['puntos'].rank(method='dense', ascending=False)  # podio
df['percentil'] = df['nota'].rank(pct=True)                          # 0..1
```

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 3</b>

Construye un ranking de **alumnos por nota descendente** sobre este DataFrame:

```python
notas = pd.DataFrame({
    'alumno': ['Ana', 'Bea', 'Carlos', 'David', 'Eva'],
    'nota':   [8.5,    9.0,   7.0,      9.0,    8.5]
})
```

1. Anade una columna `posicion` que de la posicion del alumno usando `method='dense'` (sin huecos en empates).
2. Anade otra columna `percentil` con `pct=True`.
3. Ordena el resultado por `posicion` ascendente.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd

notas = pd.DataFrame({
    'alumno': ['Ana', 'Bea', 'Carlos', 'David', 'Eva'],
    'nota':   [8.5,    9.0,   7.0,      9.0,    8.5]
})



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
#
# notas = pd.DataFrame({
#     'alumno': ['Ana', 'Bea', 'Carlos', 'David', 'Eva'],
#     'nota':   [8.5,    9.0,   7.0,      9.0,    8.5]
# })
#
# # rank descendente: el de mas nota recibe posicion 1
# notas['posicion']  = notas['nota'].rank(method='dense', ascending=False)
# notas['percentil'] = notas['nota'].rank(pct=True)
#
# display(notas.sort_values('posicion'))


In [ ]:
display(frame)

In [ ]:
display(frame)
# display(frame.rank())
# display(frame.rank(axis=1))
display(frame.rank(method='average', axis=1))

# Operaciones

Operaciones matemáticas entre objetos

In [ ]:
matrixA = np.random.randint(100,size=(4,4))
matrixB = np.random.randint(100,size=(4,4))
frameA = pd.DataFrame(matrixA)
frameB = pd.DataFrame(matrixB)
display(frameA)
display(frameB)

In [ ]:
# a través de métodos u operadores
display(frameA + frameB == frameA.add(frameB))
display(frameA + frameB)

In [ ]:
display(frameB - frameA == frameB.sub(frameA))
display(frameB - frameA)

In [ ]:
# si los frames no son iguales, valor por defecto NaN
frameC = pd.DataFrame(np.random.randint(100,size=(3,3)))
display(frameA)
display(frameC)
display(frameC + frameA)

In [ ]:
# se puede especificar el valor por defecto con el argumento fill_value
display(frameA.add(frameC, fill_value=0))

Operadores aritméticos solo válidos en elementos aceptables

In [ ]:
frameD = pd.DataFrame({0: ['a','b'],1:['d','f']})
display(frameD)
frameA - frameD

Operaciones entre Series y DataFrames

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Broadcasting: operaciones entre Series y DataFrames</b>

Cuando haces `df - serie`, Pandas hace **broadcasting**: aplica la operacion **fila a fila** o **columna a columna** segun como alineen las etiquetas.

#### Regla general
- Si la Serie tiene los **mismos nombres que las COLUMNAS** del DataFrame -> se aplica **fila a fila** (defecto).
- Si la Serie tiene los **mismos nombres que las FILAS** del DataFrame -> usa el argumento `axis=0`.

```python
df = pd.DataFrame({'A':[1,2,3], 'B':[10,20,30]})

# Serie cuyo indice coincide con las COLUMNAS de df:
s_col = pd.Series({'A':1, 'B':10})
df - s_col           # se resta a CADA fila
#    A   B
# 0  0   0
# 1  1  10
# 2  2  20

# Serie cuyo indice coincide con las FILAS de df:
s_fila = pd.Series([100, 200, 300])
df.sub(s_fila, axis=0)   # se resta a CADA columna
```

#### Caso muy comun: centrar (restar la media)
```python
df_centrado = df - df.mean()         # df.mean() es Serie con index = columnas
```

#### Caso muy comun: normalizar
```python
df_norm = (df - df.mean()) / df.std()  # z-score
```

> Sin esta alineacion automatica, tendrias que hacer un bucle por columna. Con Pandas es una linea.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 4</b>

Tienes un DataFrame de notas:

```python
df = pd.DataFrame({
    'mate':   [7, 8, 9, 6],
    'lengua': [9, 7, 5, 8],
    'arte':   [10, 9, 7, 6]
}, index=['Ana', 'Bea', 'Carlos', 'David'])
```

1. Calcula la **media** de cada asignatura (Serie con `df.mean()`).
2. Resta la media a cada nota para obtener un DataFrame **centrado** (cada celda dice cuanto se aleja de la media de su asignatura).
3. Calcula tambien las notas **normalizadas (z-score)**: `(df - df.mean()) / df.std()`.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd

df = pd.DataFrame({
    'mate':   [7, 8, 9, 6],
    'lengua': [9, 7, 5, 8],
    'arte':   [10, 9, 7, 6]
}, index=['Ana', 'Bea', 'Carlos', 'David'])



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
#
# df = pd.DataFrame({
#     'mate':   [7, 8, 9, 6],
#     'lengua': [9, 7, 5, 8],
#     'arte':   [10, 9, 7, 6]
# }, index=['Ana', 'Bea', 'Carlos', 'David'])
#
# # 1) media por columna
# medias = df.mean()
# print(medias)
#
# # 2) centrado: la media de cada columna a cero
# df_centrado = df - medias
# display(df_centrado)
#
# # 3) z-score: media 0, std 1
# df_z = (df - df.mean()) / df.std()
# display(df_z)


In [ ]:
rand_matrix = np.random.randint(10, size=(3, 4))
df = pd.DataFrame(rand_matrix , columns=list('ABCD'))
display(df)

In [ ]:
primera_fila = df.iloc[0, :]
primera_fila.name = "Primera fila"
print(primera_fila)
print()

# uso común, averiguar la diferencia entre una fila y el resto
# display(df - primera_fila)
display(df.sub(primera_fila, axis=1))

In [ ]:
# # Por columnas cómo se restaría
primera_columna = df['A']
primera_columna.name = "Primera columna"
print(primera_columna)
print()

display(df.sub(primera_columna, axis=0))

pandas se basa en NumPy, np operadores binarios y unarios son aceptables 

| Tipo | Operación | Descripción |
|:---------|:-----|:-----|
| Unario | *abs* | Valor absoluto de cada elemento |
| | *sqrt* | Raíz cuadrada de cada elemento |
| | *exp* | e^x, siendo x cad elemento |
| | *log, log10, log2* | Logaritmos en distintas bases de cada elemento |
| | *sign* | Retorna el signo de cada elemento (-1 para negativo, 0 o 1 para positivo) |
| | *ceil* | Redondea cada elemento por arriba |
| | *floor* | Redondea cada elemento por abajo |
| | *isnan* | Retorna si cada elemento es Nan |
| | *cos, sin, tan* | Operaciones trigonométricas en cada elemento |
| | *arccos, arcsin, arctan* | Inversas de operaciones trigonométricas en cada elemento |
| Binario | *add* | Suma de dos arrays |
| | *substract* | Resta de dos arrays |
| | *multiply* | Multiplicación de dos arrays |
| | *divide* | División de dos arrays |
| | *maximum, minimum* | Retorna el valor máximo/mínimo de cada pareja de elementos |
| | *equal, not_equal* | Retorna la comparación (igual o no igual) de cada pareja de elementos |
| | *greater, greater_equal, less, less_equal* | Retorna la comparación (>, >=, <, <= respectivamente) de cada pareja de elementos |

Si queremos aplicar una función a cada columna (o fila) utilizamos `apply`

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; apply: aplicar una funcion a cada fila o columna</b>

`apply` es la **navaja suiza** de Pandas: ejecuta una funcion sobre **cada columna** (o cada fila) del DataFrame y junta los resultados.

```python
df.apply(funcion)            # axis=0 (defecto): la funcion recibe cada COLUMNA
df.apply(funcion, axis=1)    # la funcion recibe cada FILA
```

#### Tres patrones tipicos

**1. Estadistica por columna** (cuando un metodo nativo no existe):
```python
df.apply(lambda col: col.max() - col.min())   # rango de cada columna
```

**2. Estadistica por fila**:
```python
df.apply(lambda fila: fila.sum(), axis=1)    # suma de cada fila
```

**3. Crear una columna nueva combinando varias**:
```python
df['IMC'] = df.apply(lambda fila: fila['peso'] / fila['altura']**2, axis=1)
```

#### apply vs map vs applymap (cuando usar cada uno)

| Quien lo tiene | Funcion | Sirve para |
|---|---|---|
| Serie | `s.map(f)` | Aplicar `f` a **cada elemento** de la Serie. |
| DataFrame | `df.apply(f)` | Aplicar `f` a **cada fila o columna** (recibe Serie). |
| DataFrame | `df.applymap(f)` (o `.map` en pandas 2.1+) | Aplicar `f` a **cada celda** individual. |

> **Rendimiento:** apply es flexible pero NO es el codigo mas rapido. Si puedes resolver tu problema con operaciones vectorizadas (`df['a'] + df['b']`), siempre seran mas rapidas que `apply`. Usa `apply` cuando NO existe la version vectorizada.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 5</b>

Tienes este DataFrame de personas con peso (kg) y altura (m):

```python
personas = pd.DataFrame({
    'nombre':  ['Ana', 'Bea', 'Carlos', 'David'],
    'peso':    [60, 75, 85, 70],
    'altura':  [1.65, 1.80, 1.75, 1.70]
})
```

1. Usando `apply` con `axis=1`, crea una columna `IMC` = `peso / altura**2`.
2. Usando `apply` (sin axis) sobre las columnas numericas (`peso`, `altura`), calcula el **rango** (`max - min`) de cada una.
3. Crea otra columna `categoria` con `apply(axis=1)`: `'bajo'` si IMC<18.5, `'normal'` si IMC<25, `'alto'` en otro caso.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd

personas = pd.DataFrame({
    'nombre':  ['Ana', 'Bea', 'Carlos', 'David'],
    'peso':    [60, 75, 85, 70],
    'altura':  [1.65, 1.80, 1.75, 1.70]
})



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
#
# personas = pd.DataFrame({
#     'nombre':  ['Ana', 'Bea', 'Carlos', 'David'],
#     'peso':    [60, 75, 85, 70],
#     'altura':  [1.65, 1.80, 1.75, 1.70]
# })
#
# # 1) IMC por fila
# personas['IMC'] = personas.apply(lambda r: r['peso'] / r['altura']**2, axis=1)
#
# # 2) rango por columna numerica
# rangos = personas[['peso', 'altura']].apply(lambda c: c.max() - c.min())
# print(rangos)
#
# # 3) categoria por fila
# def categoria(r):
#     if r['IMC'] < 18.5:
#         return 'bajo'
#     elif r['IMC'] < 25:
#         return 'normal'
#     else:
#         return 'alto'
#
# personas['categoria'] = personas.apply(categoria, axis=1)
# display(personas)


In [ ]:
rand_matrix = np.random.randint(10, size=(3, 4))
frame = pd.DataFrame(rand_matrix , columns=list('ABCD'))
display(frame)

In [ ]:
def max_min(x):
    return x.max() - x.min()

serie = pd.Series()
serie.name = "Diferencia max min"
for col in frame:
    serie[col] = max_min(frame[col])
    
print(serie)
print()

serie2 = frame.apply(max_min) # diferencia por columna
serie2.name = "Diferencia max min usando apply"
print(serie2)

In [ ]:
# Las funciones lambda están pensadas para estos casos
serie3 = frame.apply(lambda x : x.max() - x.min()) # diferencia por columna
serie3.name = "Diferencia max min usando apply y función lambda"
print(serie3) 

In [ ]:
# El método apply también puede aplicarse por filas definiendo el eje de iteración
serie4 = frame.apply(lambda x : x.max() - x.min(), axis=1)
serie4.name = "Diferencia max min usando apply, función lambda y por filas"
print(serie4)

# Estadística descriptiva
- Análisis preliminar de los datos
- Para Series y DataFrame

| Operación | Descripción |
|:----------|:------------|
| count | Número de valores no NaN |
| describe | Conjunto de estadísticas sumarias|
| min, max | Valores mínimo y máximo |
| argmin, argmax | Índices posicionales del valor mínimo y máximo |
| idxmin, idxmax | Índices semánticos del valor mínimo y máximo |
| sum | Suma de los elementos |
| mean | Media de los elementos |
| median | Mediana de los elementos |
| mad | Desviación absoluta media del valor medio |
| var | Varianza de los elementos |
| std | Desviación estándar de los elementos |
| cumsum | Suma acumulada de los elementos |
| diff | Diferencia aritmética de los elementos |

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Estadistica descriptiva: la chuleta del analista</b>

Antes de modelar nada, **mira tus datos**. Pandas tiene metodos vectorizados para todos los estadisticos basicos.

#### Metodos generales sobre DataFrame / Serie

| Metodo | Devuelve |
|---|---|
| `count()` | Numero de **valores no nulos** por columna. |
| `sum()` | Suma. |
| `mean()` | Media aritmetica. |
| `median()` | Mediana (valor central). |
| `std()` | Desviacion tipica (n-1). |
| `var()` | Varianza. |
| `min()`, `max()` | Min y max. |
| `idxmin()`, `idxmax()` | **Indice** (etiqueta) del min/max, no el valor. |
| `quantile(0.25)` | Cuantiles. |
| `cumsum()`, `cummax()`, `cumprod()` | Acumulados. |
| `describe()` | Resumen completo: count, mean, std, min, 25%, 50%, 75%, max. |
| `value_counts()` | Frecuencia de cada valor (en Series). |
| `nunique()` | Cuantos valores **distintos** hay. |
| `corr()` | Matriz de correlaciones. |
| `cov()` | Matriz de covarianzas. |

#### Argumentos utiles
- `axis=0` (defecto, columnas) o `axis=1` (filas).
- `numeric_only=True`: ignora columnas no numericas en lugar de fallar.
- `skipna=True` (defecto): ignora NaN. Cambialo a `False` si quieres que un solo NaN haga el resultado NaN.

#### `describe()` segun el tipo
```python
df.describe()                   # solo numericas (defecto)
df.describe(include='all')      # tambien object/categorical (count, unique, top, freq)
df.describe(include='object')   # solo categoricas
```

#### Truco: encadenar para top-N
```python
df['ciudad'].value_counts().head(5)   # las 5 ciudades mas frecuentes
```

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 6</b>

Tienes el DataFrame del notebook:

```python
diccionario = {
    'nombre': ['Marisa', 'Laura', 'Manuel', 'Carlos'],
    'edad':   [34, 34, 11, 30],
    'puntos': [95, 80, 50, 50]
}
frame = pd.DataFrame(diccionario)
```

Calcula:
1. La **media** y la **mediana** de la columna `puntos`.
2. La **edad maxima** y el **nombre** de la persona con esa edad (pista: `idxmax`).
3. El **percentil 25** (cuartil inferior) de `puntos`.
4. Un **resumen completo** con `describe(include='all')`.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd

diccionario = {
    'nombre': ['Marisa', 'Laura', 'Manuel', 'Carlos'],
    'edad':   [34, 34, 11, 30],
    'puntos': [95, 80, 50, 50]
}
frame = pd.DataFrame(diccionario)



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
#
# diccionario = {
#     'nombre': ['Marisa', 'Laura', 'Manuel', 'Carlos'],
#     'edad':   [34, 34, 11, 30],
#     'puntos': [95, 80, 50, 50]
# }
# frame = pd.DataFrame(diccionario)
#
# # 1) media y mediana de puntos
# print('Media puntos:  ', frame['puntos'].mean())
# print('Mediana puntos:', frame['puntos'].median())
#
# # 2) edad maxima y persona
# idx = frame['edad'].idxmax()           # idx es el indice de la fila
# print('Edad max: ', frame['edad'].max())
# print('Quien:    ', frame.loc[idx, 'nombre'])
#
# # 3) percentil 25
# print('Q1 puntos:', frame['puntos'].quantile(0.25))
#
# # 4) describe completo (incluye nombre)
# display(frame.describe(include='all'))


In [ ]:
diccionario = { "nombre" : ["Marisa","Laura","Manuel", "Carlos"], "edad" : [34,34,11, 30], 
               "puntos" : [98,12,68,np.nan], "genero": ["F", "F", "M", "M"] }
frame = pd.DataFrame(diccionario)
display(frame)
print("Número de elementos no NaN:")
print(frame.count())    # datos generales de elementos
print()
print("Estadística de elementos numéricos:")
display(frame.describe()) # datos generales de elementos

In [ ]:
# Podemos acceder a estos valores
edad_media = frame.describe()['edad']['mean']
print(edad_media)

In [ ]:
# operadores básicos
print(frame.sum())
print()
print("Ignorando datos no numéricos")
print(frame.sum(numeric_only=True))

In [ ]:
frame.mean(numeric_only=True)

In [ ]:
frame.cumsum()

In [ ]:
frame.count(axis=1)

In [ ]:
print(frame['edad'].std())

In [ ]:
frame['edad'].idxmax()

In [ ]:
frame['puntos'].idxmin()

In [ ]:
# frame con las filas con los valores maximos de una columna
print(frame['puntos'].max())

mask = frame['puntos'] == frame['puntos'].max()
display(frame[mask])

idm = frame['puntos'].idxmax()
display(frame.loc[idm])

In [ ]:
frame["ranking"] = frame["puntos"].rank(method='max', ascending=False)  
display(frame)

## Agregaciones

Las agregaciones son operaciones que realizamos sobre *categorías*.

Ejemplo, la columna `genero` sólo puede tener un número finito de valores (masculino "M" o femenino "F"), por lo que podemos deducir que es una columna categórica.

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; groupby: el corazon de Pandas (split-apply-combine)</b>

Si solo te llevas UNA cosa de este notebook, que sea `groupby`. Es el equivalente Pandas de `GROUP BY` en SQL y de las **tablas dinamicas** de Excel.

#### El patron split-apply-combine

`groupby` aplica un patron de tres pasos:

```
   datos
     |
     | SPLIT  -> partir en grupos segun una columna categorica
     v
   [grupo A] [grupo B] [grupo C]
     |        |        |
     | APPLY -> calcular algo en cada grupo (mean, sum, count, custom...)
     v        v        v
    valA     valB     valC
     \       |       /
      \______|______/
             |
             | COMBINE -> juntar los resultados en un nuevo DataFrame/Serie
             v
           resultado
```

#### Ejemplo guiado

Imagina ventas:

| producto | tienda | unidades |
|---|---|---|
| pan      | A | 10 |
| pan      | B | 15 |
| leche    | A | 8  |
| leche    | B | 12 |
| pan      | A | 5  |

```python
ventas.groupby('producto')['unidades'].sum()
# producto
# leche    20
# pan      30
```

Que paso por dentro:
1. **Split**: separa filas con `producto='pan'` de las de `producto='leche'`.
2. **Apply**: en cada grupo, suma la columna `unidades`.
3. **Combine**: arma una Serie indexada por producto.

#### Sintaxis general

```python
df.groupby('columna')                       # devuelve un GroupBy object (perezoso)
df.groupby('columna')['col_num'].mean()      # media de una columna por grupo
df.groupby(['col1', 'col2']).sum()           # group por VARIAS columnas (multinivel)
df.groupby('col').agg(['mean', 'std', 'count'])   # varios estadisticos a la vez
df.groupby('col').agg({'A': 'mean', 'B': 'sum'})  # estadisticos distintos por columna
```

#### Funciones de agregacion comunes

| Funcion | Que hace |
|---|---|
| `count()` | Numero de elementos no nulos. |
| `size()` | Numero de elementos (incluyendo NaN). |
| `sum()`, `mean()`, `median()`, `min()`, `max()`, `std()`, `var()` | Las habituales. |
| `first()`, `last()`, `nth(n)` | Primer / ultimo / n-esimo elemento del grupo. |
| `nunique()` | Cuantos distintos hay por grupo. |
| `agg(...)` | Aplicar una o varias funciones a la vez. |

#### transform vs apply vs filter (en groupby)

| Metodo | Devuelve | Para que |
|---|---|---|
| `agg` / metodos | Una fila por grupo. | Resumir. |
| `transform` | Mismo tamano que el original. | Anadir una columna calculada por grupo (ej. media del grupo en cada fila). |
| `filter` | Mismo formato, menos filas. | Quedarte solo con los grupos que cumplen una condicion. |
| `apply` | Lo que tu quieras. | Mas flexible, mas lento. |

```python
# transform: anadir la media del grupo a cada fila original
df['media_tienda'] = df.groupby('tienda')['unidades'].transform('mean')

# filter: quedarte con grupos que tengan al menos 100 unidades en total
df.groupby('tienda').filter(lambda g: g['unidades'].sum() > 100)
```

> **Excel mental model:** `groupby` + `agg` es practicamente una **tabla dinamica**: filas = grupos, columnas = estadisticos.

</div>

<div style="background-color:#f3f0ff;border-left:6px solid #7048e8;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#4c2d9e;font-size:14px;">PARA RECORDAR</b>

**Notacion mental que ayuda mucho:**

```python
df.groupby('X')['Y'].FUNCION()
#       ^         ^         ^
#       |         |         |
#       columna   columna   que calcular en cada grupo
#       que crea  que mide  (mean, sum, count...)
#       grupos
```

Lee como: *"Para cada valor de X, calcula FUNCION sobre Y"*.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 7</b>

Tienes ventas por tienda y producto:

```python
ventas = pd.DataFrame({
    'tienda':   ['A', 'A', 'B', 'B', 'A', 'B', 'A', 'B'],
    'producto': ['pan', 'leche', 'pan', 'queso', 'queso', 'leche', 'pan', 'queso'],
    'unidades': [10, 5, 7, 3, 6, 4, 2, 8],
    'precio':   [1.2, 0.95, 1.2, 4.3, 4.3, 0.95, 1.2, 4.3]
})
```

Calcula:

1. **Total de unidades vendidas por tienda**.
2. **Media de unidades por producto**.
3. **Total de unidades** y **media de precio** por (`tienda`, `producto`) usando un solo `agg(...)`.
4. Anade una columna `pct_tienda` con el **porcentaje** que cada fila representa sobre el total de su tienda (pista: `transform('sum')`).
5. Quedate solo con las tiendas cuya **suma total** de unidades sea **mayor de 20** (pista: `filter`).

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd

ventas = pd.DataFrame({
    'tienda':   ['A', 'A', 'B', 'B', 'A', 'B', 'A', 'B'],
    'producto': ['pan', 'leche', 'pan', 'queso', 'queso', 'leche', 'pan', 'queso'],
    'unidades': [10, 5, 7, 3, 6, 4, 2, 8],
    'precio':   [1.2, 0.95, 1.2, 4.3, 4.3, 0.95, 1.2, 4.3]
})
display(ventas)



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
#
# ventas = pd.DataFrame({
#     'tienda':   ['A', 'A', 'B', 'B', 'A', 'B', 'A', 'B'],
#     'producto': ['pan', 'leche', 'pan', 'queso', 'queso', 'leche', 'pan', 'queso'],
#     'unidades': [10, 5, 7, 3, 6, 4, 2, 8],
#     'precio':   [1.2, 0.95, 1.2, 4.3, 4.3, 0.95, 1.2, 4.3]
# })
#
# # 1) total de unidades por tienda
# print(ventas.groupby('tienda')['unidades'].sum())
#
# # 2) media de unidades por producto
# print(ventas.groupby('producto')['unidades'].mean())
#
# # 3) varias agregaciones a la vez por (tienda, producto)
# resumen = ventas.groupby(['tienda', 'producto']).agg(
#     unidades_tot=('unidades', 'sum'),
#     precio_medio=('precio', 'mean')
# )
# display(resumen)
#
# # 4) porcentaje sobre el total de la tienda usando transform
# total_tienda = ventas.groupby('tienda')['unidades'].transform('sum')
# ventas['pct_tienda'] = ventas['unidades'] / total_tienda * 100
# display(ventas)
#
# # 5) filter: solo tiendas con suma de unidades > 20
# resultado = ventas.groupby('tienda').filter(lambda g: g['unidades'].sum() > 20)
# display(resultado)


In [ ]:
# El método groupby() se encarga de dividir los datos por categorías según la columna que se indique.
display(frame)
datos_por_genero = frame.groupby('genero')
print(datos_por_genero) # Es un elemento no realizado esperando por una función de agregación

In [ ]:
# En este caso, la función de agregación es count(), que se aplicará a cada categoría
# count_por_genero = datos_por_genero.count()
# display(count_por_genero)

# Otras funciones de agregación
display(datos_por_genero.mean(numeric_only=True))
display(datos_por_genero.max())

In [ ]:
# podemos incluso obtener distintos estadísticos con el método aggregate
display(datos_por_genero[['edad', 'puntos']].aggregate(['min', 'mean', 'max']))

In [ ]:
# El método filter te permite hacer operaciones sobre los grupos y descartar aquellos que no cumplan una condición determinada
frame2 = datos_por_genero.filter(lambda x : x['edad'].mean() > 30)
display(frame2)

In [ ]:
# Sin utilizar lambda sería:
def media(x):
    display(x) # x será un dataframe
    return x["edad"].mean() > 30

frame3 = datos_por_genero.filter(media)
# display(frame3)

In [ ]:
# No hay categorías Nan, el groupby los descarta por defecto
df = frame.groupby('puntos').count()
display(df)

## Correlaciones

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Correlaciones: medir la relacion entre dos variables</b>

La **correlacion** mide si dos variables numericas se mueven juntas. La mas usada es la **correlacion de Pearson**, que da un numero entre **-1 y 1**:

| Valor | Significado |
|---|---|
| `+1` | Relacion lineal **perfecta y directa** (sube una, sube la otra). |
| `0`  | No hay relacion lineal. |
| `-1` | Relacion lineal **perfecta e inversa** (sube una, baja la otra). |

Y los valores intermedios:

| Rango | Interpretacion (orientativa) |
|---|---|
| 0.0 - 0.3 | Debil |
| 0.3 - 0.7 | Moderada |
| 0.7 - 1.0 | Fuerte |

> **Cuidado:** correlacion **NO implica causalidad**. Que dos cosas suban juntas no significa que una cause la otra (puede haber una tercera variable detras).

#### En Pandas

```python
df['x'].corr(df['y'])         # correlacion entre dos columnas (Pearson)
df.corr(numeric_only=True)    # matriz de correlacion entre TODAS las numericas
df.corr(method='spearman')    # rangos -> mide relacion monotona, no solo lineal
df.corr(method='kendall')     # otra alternativa robusta
```

#### Diferencia entre Pearson y Spearman
- **Pearson**: mide relacion **lineal**.
- **Spearman**: convierte cada valor a su **ranking** y mide relacion. Robusto a outliers y captura relaciones monotonas no lineales (ej. exponencial creciente).

#### Por que importa para Machine Learning
- **Feature selection**: si dos features tienen correlacion 0.99, son practicamente la misma informacion -> eliminar una.
- **Multicolinealidad**: en regresion lineal, features muy correlacionadas hacen los coeficientes inestables.
- **Visualizacion**: heatmaps de correlaciones son lo primero que mira un data scientist.

</div>

pandas incluye métodos para analizar correlaciones
- Relación matemática entre dos variables (-1 negativamente relacionadas, 1 positivamente relacionadas, 0 sin relación)
- obj.corr(obj2) --> medida de correlación entre los datos de ambos objetos

### Ejemplo Fuel efficiency
- https://archive.ics.uci.edu/ml/datasets/Auto+MPG
- Objetivo:
  - Determinar las correlación del milleage per gallons (mpg) de los coches respecto a otras especificaciones técnicas

In [ ]:
import pandas as pd
path = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'

mpg_data = pd.read_csv(path, sep='\\s+', header=None,
            names = ['mpg', 'cilindros', 'desplazamiento','potencia',
            'peso', 'aceleracion', 'año', 'origen', 'nombre'],
            na_values='?', engine='c')

In [ ]:
display(mpg_data.sample(5))
mpg_data.info()

In [ ]:
display(mpg_data.describe(include='all'))

### Correlaciones entre valores

In [ ]:
mpg_data['mpg'].corr(mpg_data['peso']) # + mpg = - peso

In [ ]:
mpg_data['peso'].corr(mpg_data['aceleracion']) # + peso = - aceleracion

### Correlaciones entre todos los valores

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 8</b>

Carga el dataset `mpg_data` que ya ha cargado el notebook (consumo de coches).

1. Calcula la matriz de correlacion entre todas las columnas numericas (`mpg_data.corr(numeric_only=True)`).
2. Encuentra el **par de columnas con correlacion mas negativa** (excluyendo la diagonal).
3. Encuentra el **par con correlacion mas positiva** (excluyendo la diagonal).
4. Para la columna `mpg`, ordena las demas columnas por **correlacion absoluta descendente** (las que mas la explican, en cualquier sentido).

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
# nota: requiere que el notebook ya haya cargado mpg_data en celdas anteriores



In [ ]:
# === Solucion sugerida ===
# # 1) matriz de correlacion
# import numpy as np
# corr = mpg_data.corr(numeric_only=True)
# display(corr)
#
# # 2) y 3): excluir la diagonal poniendola a NaN para que min/max no la consideren
# corr_off = corr.where(~np.eye(len(corr), dtype=bool))
#
# # par mas negativo:
# # idxmin sobre cada columna devuelve la fila con min, despues coger el min global
# valor_min = corr_off.min().min()
# # buscar (fila, col) con ese valor
# pos = corr_off.stack().idxmin()       # tupla (fila, col)
# print('Mas negativo:', pos, '->', valor_min)
#
# # par mas positivo:
# pos_pos = corr_off.stack().idxmax()
# print('Mas positivo:', pos_pos, '->', corr_off.stack().max())
#
# # 4) lo que mas explica mpg, en valor absoluto
# corr_mpg = corr['mpg'].drop('mpg')
# print(corr_mpg.abs().sort_values(ascending=False))


In [ ]:
mpg_data.corr(numeric_only=True)

In [ ]:
#año y origen no parecen correlacionables
#eliminar columnas de la correlacion
corr_data = mpg_data.drop(['año','origen'],axis=1).corr(numeric_only=True)
display(corr_data)

In [ ]:
# representación gráfica matplotlib
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# representación gráfica
corr_data.style.background_gradient(cmap=plt.get_cmap('RdYlGn'), axis=1, vmin=-1, vmax=1)

In [ ]:
# correlación más negativa
corr_data.idxmin()

In [ ]:
# correlación más positiva
corr_data.idxmax()  #consigo misma....

In [ ]:
# tabla similar con las correlaciones más positivas (evitar parejas del mismo valor)
corr_data = corr_data.where(~np.eye(len(corr_data), dtype=bool))

# Ya no tenemos la diagonal igual a 1
display(corr_data)
corr_data.idxmax()

In [ ]:
corr_data.style.background_gradient(cmap=plt.get_cmap('RdYlGn'), axis=1, vmin=-1, vmax=1)

In [ ]:
mpg_corr = corr_data['mpg'].drop('mpg')
mpg_corr.name = "Correlaciones del mpg respecto a las otras característica"
print(mpg_corr)

## Ejercicios

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Recapitulacion del notebook 10</b>

Has visto los pilares de Pandas para analisis de datos:

1. **Operaciones vectorizadas** entre DataFrames y Series (con alineacion por etiquetas).
2. **`apply`** para funciones personalizadas por fila o columna.
3. **Estadistica descriptiva** (`describe`, `mean`, `std`, `quantile`, `idxmax`).
4. **`groupby`**: split-apply-combine para resumir por categorias.
5. **Correlaciones**: medir relaciones lineales entre variables numericas.

#### Que falta (y veras en proximos materiales)
- **`merge` / `join` / `concat`**: combinar varios DataFrames (equivalente a SQL JOIN).
- **`pivot` / `melt`**: cambiar el formato entre **ancho** (columnas por categoria) y **largo** (una fila por observacion).

#### Una vista rapida de merge / pivot / melt (adelanto)

**merge** combina dos DataFrames por una columna comun:
```python
clientes.merge(pedidos, on='id_cliente', how='inner')
# how='inner' (defecto), 'left', 'right', 'outer'
```

| `how`     | Filas resultantes |
|---|---|
| `'inner'` | Solo las claves que estan en LOS DOS. |
| `'left'`  | Todas las del izquierdo, NaN cuando no hay derecho. |
| `'right'` | Todas las del derecho. |
| `'outer'` | Todas las de los dos lados, NaN donde falte. |

**pivot** convierte filas en columnas (formato ancho):
```python
ventas.pivot(index='fecha', columns='producto', values='unidades')
```

**melt** hace lo contrario (formato largo):
```python
df.melt(id_vars=['fecha'], var_name='producto', value_name='unidades')
```

| Excel | Pandas |
|---|---|
| Tabla dinamica | `groupby` + `agg`, o `pivot_table` |
| BUSCARV / VLOOKUP | `merge` |
| Apilar columnas | `melt` |
| Cruzar tabla | `pivot` |

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 9</b>

**Ejercicio final integrador.** Tienes dos DataFrames:

```python
clientes = pd.DataFrame({
    'id_cliente': [1, 2, 3, 4],
    'nombre':     ['Ana', 'Bea', 'Carlos', 'David'],
    'ciudad':     ['Madrid', 'Madrid', 'Bilbao', 'Sevilla']
})

pedidos = pd.DataFrame({
    'id_pedido':  [101, 102, 103, 104, 105, 106],
    'id_cliente': [1, 2, 1, 3, 1, 2],
    'producto':   ['libro', 'taza', 'libro', 'lampara', 'taza', 'libro'],
    'importe':    [15.0, 8.0, 22.0, 35.0, 9.0, 12.0]
})
```

1. Haz un **`merge`** de `pedidos` con `clientes` (`how='left'` sobre `id_cliente`).
2. Sobre el resultado, agrupa por `ciudad` y calcula la **suma del importe** y el **numero de pedidos**.
3. Ordena el resultado por la suma del importe descendente.
4. **Bonus**: usando `pivot_table`, crea una tabla con filas = `ciudad`, columnas = `producto`, valores = suma del importe (rellena los huecos con 0).

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd

clientes = pd.DataFrame({
    'id_cliente': [1, 2, 3, 4],
    'nombre':     ['Ana', 'Bea', 'Carlos', 'David'],
    'ciudad':     ['Madrid', 'Madrid', 'Bilbao', 'Sevilla']
})

pedidos = pd.DataFrame({
    'id_pedido':  [101, 102, 103, 104, 105, 106],
    'id_cliente': [1, 2, 1, 3, 1, 2],
    'producto':   ['libro', 'taza', 'libro', 'lampara', 'taza', 'libro'],
    'importe':    [15.0, 8.0, 22.0, 35.0, 9.0, 12.0]
})



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
#
# clientes = pd.DataFrame({
#     'id_cliente': [1, 2, 3, 4],
#     'nombre':     ['Ana', 'Bea', 'Carlos', 'David'],
#     'ciudad':     ['Madrid', 'Madrid', 'Bilbao', 'Sevilla']
# })
#
# pedidos = pd.DataFrame({
#     'id_pedido':  [101, 102, 103, 104, 105, 106],
#     'id_cliente': [1, 2, 1, 3, 1, 2],
#     'producto':   ['libro', 'taza', 'libro', 'lampara', 'taza', 'libro'],
#     'importe':    [15.0, 8.0, 22.0, 35.0, 9.0, 12.0]
# })
#
# # 1) merge: cada pedido se completa con la info del cliente
# combinado = pedidos.merge(clientes, on='id_cliente', how='left')
# display(combinado)
#
# # 2) agregacion por ciudad
# resumen = combinado.groupby('ciudad').agg(
#     importe_total=('importe', 'sum'),
#     pedidos=('id_pedido', 'count')
# )
#
# # 3) ordenar por importe descendente
# resumen = resumen.sort_values('importe_total', ascending=False)
# display(resumen)
#
# # 4) tabla pivote: ciudad x producto, valores = suma de importe
# tabla = combinado.pivot_table(
#     index='ciudad',
#     columns='producto',
#     values='importe',
#     aggfunc='sum',
#     fill_value=0
# )
# display(tabla)


- Ejercicios para practicar Pandas: https://github.com/ajcr/100-pandas-puzzles/blob/master/100-pandas-puzzles.ipynb
- Soluciones en: https://github.com/ajcr/100-pandas-puzzles/blob/master/100-pandas-puzzles-with-solutions.ipynb